# 🔍 Active Visual Search - Environment Demo

This notebook demonstrates the visual search environment and core components.

**What you'll learn:**
- How the environment works
- How to create search scenarios
- How the agent observes and acts
- How to visualize search trajectories

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('../')  # Add project root to path

import torch
import torchvision
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import yaml

# Import our custom modules
from src.environment import VisualSearchEnv
from src.models.dinov3_encoder import DINOv3Encoder

# Set up matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Configuration and Dataset

In [ ]:
# Load configuration
with open('../config/default_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  Canvas size: {config['environment']['canvas_size']}")
print(f"  Window size: {config['environment']['window_size']}")
print(f"  Max steps: {config['environment']['max_steps']}")
print(f"  Num objects: {config['environment']['num_objects']}")

In [ ]:
# Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor()
])

cifar10 = datasets.CIFAR10(
    root='../data',
    train=True,
    download=True,
    transform=transform
)

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"\n✅ CIFAR-10 loaded: {len(cifar10)} images")
print(f"Target classes: {[class_names[i] for i in config['dataset']['target_classes']]}")

## 3. Visualize CIFAR-10 Samples

In [ ]:
# Show some CIFAR-10 samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('CIFAR-10 Sample Images', fontsize=16)

for idx, target_class in enumerate(config['dataset']['target_classes']):
    # Find first image of this class
    for i, (img, label) in enumerate(cifar10):
        if label == target_class:
            ax = axes[idx // 5, idx % 5]
            ax.imshow(img.permute(1, 2, 0))
            ax.set_title(class_names[target_class])
            ax.axis('off')
            break

plt.tight_layout()
plt.show()

## 4. Create Environment

In [ ]:
# Create environment
device = 'cuda' if torch.cuda.is_available() else 'cpu'

env = VisualSearchEnv(
    config=config,
    dataset=cifar10,
    device=device
)

print("✅ Environment created!")
print(f"Action space: {env.action_space_n} actions")
print("Actions: 0=Up, 1=Down, 2=Left, 3=Right, 4=Found")

## 5. Test Environment - Single Episode

In [ ]:
# Reset environment
state = env.reset()

print("Episode initialized!")
print(f"\nState information:")
print(f"  Observation shape: {state['observation'].shape}")
print(f"  Position: {state['position']}")
print(f"  Target class: {class_names[state['target_class']]}")
print(f"  Steps taken: {state['steps_taken']}")

env_info = env.get_info()
print(f"\nEnvironment info:")
print(f"  Target position: {env_info['target_position']}")
print(f"  Agent position: {env_info['agent_position']}")
print(f"  Objects on canvas: {env_info['num_objects']}")
print(f"  Object classes: {[class_names[c] for c in env_info['object_classes']]}")

In [ ]:
# Visualize initial state
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Full canvas with agent viewport
canvas_render = env.render()
axes[0].imshow(canvas_render)
axes[0].set_title(f'Full Canvas (Target: {class_names[state["target_class"]]})', fontsize=14)
axes[0].set_xlabel('Red box = Agent viewport, Blue circle = Target', fontsize=10)
axes[0].axis('off')

# Agent's current observation
obs_img = state['observation'].permute(1, 2, 0)
# Denormalize for visualization
obs_img = obs_img * torch.tensor([0.229, 0.224, 0.225]) + torch.tensor([0.485, 0.456, 0.406])
obs_img = torch.clamp(obs_img, 0, 1)

axes[1].imshow(obs_img.numpy())
axes[1].set_title("Agent's Current View (64x64)", fontsize=14)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 6. Simulate Random Agent

In [ ]:
# Run episode with random actions
state = env.reset()
done = False
episode_reward = 0
frames = []

print(f"Starting episode - Target: {class_names[state['target_class']]}\n")

step = 0
while not done and step < 20:  # Limit to 20 steps for demo
    # Random action
    action = np.random.randint(0, 5)
    action_names = ['Up', 'Down', 'Left', 'Right', 'Found']
    
    # Step environment
    next_state, reward, done, info = env.step(action)
    episode_reward += reward
    
    # Save frame
    frames.append(env.render())
    
    print(f"Step {step+1}: Action={action_names[action]}, Reward={reward:.2f}, Done={done}")
    
    state = next_state
    step += 1

print(f"\nEpisode finished!")
print(f"Total reward: {episode_reward:.2f}")
print(f"Success: {info.get('success', False)}")
print(f"Reason: {info.get('reason', 'unknown')}")

In [ ]:
# Visualize trajectory
if len(frames) > 0:
    fig, axes = plt.subplots(2, min(5, len(frames)), figsize=(20, 8))
    if len(frames) < 5:
        axes = axes.reshape(-1, len(frames))
    
    fig.suptitle('Agent Search Trajectory (First 10 steps)', fontsize=16)
    
    for i in range(min(10, len(frames))):
        row = i // 5
        col = i % 5
        axes[row, col].imshow(frames[i])
        axes[row, col].set_title(f'Step {i+1}', fontsize=10)
        axes[row, col].axis('off')
    
    # Hide unused subplots
    for i in range(min(10, len(frames)), 10):
        row = i // 5
        col = i % 5
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

## 7. Test DINOv3 Encoder

In [ ]:
# Initialize DINOv3 encoder
print("Loading DINOv3 encoder... (this may take a minute)\n")

encoder = DINOv3Encoder(
    model_name=config['dinov3']['variant'],
    freeze=config['dinov3']['freeze'],
    use_fp16=config['dinov3']['use_fp16'],
    device=device
)

print("\n✅ DINOv3 encoder loaded!")
model_info = encoder.get_model_info()
for key, value in model_info.items():
    print(f"  {key}: {value}")

In [ ]:
# Extract features from agent's observation
state = env.reset()
obs = state['observation'].unsqueeze(0).to(device)

print(f"Input observation shape: {obs.shape}")

# Extract features
features = encoder.encode(obs)

print(f"DINOv3 features shape: {features.shape}")
print(f"Feature statistics:")
print(f"  Mean: {features.mean().item():.4f}")
print(f"  Std: {features.std().item():.4f}")
print(f"  Min: {features.min().item():.4f}")
print(f"  Max: {features.max().item():.4f}")

In [ ]:
# Compare features from different object classes
print("Comparing DINOv3 features across object classes...\n")

# Get one image from each target class
class_features = {}
class_images = {}

for target_class in config['dataset']['target_classes']:
    for i, (img, label) in enumerate(cifar10):
        if label == target_class:
            class_images[target_class] = img
            # Extract features
            img_batch = img.unsqueeze(0).to(device)
            feat = encoder.encode(img_batch)
            class_features[target_class] = feat
            break

# Compute pairwise similarities
print("Cosine similarity matrix:\n")
print(" " * 12 + "  ".join([class_names[c][:4] for c in config['dataset']['target_classes']]))

for i, class1 in enumerate(config['dataset']['target_classes']):
    print(f"{class_names[class1]:10s}", end="  ")
    for class2 in config['dataset']['target_classes']:
        sim = encoder.compute_similarity(
            class_features[class1],
            class_features[class2],
            metric='cosine'
        )
        print(f"{sim.item():.3f}", end="  ")
    print()

print("\n(1.0 = identical, 0.0 = orthogonal, -1.0 = opposite)")

## 8. Benchmark Random Agent Performance

In [ ]:
# Run multiple episodes to get baseline performance
num_test_episodes = 50
results = {
    'success': [],
    'steps': [],
    'rewards': []
}

print(f"Running {num_test_episodes} episodes with random agent...\n")

for episode in range(num_test_episodes):
    state = env.reset()
    done = False
    episode_reward = 0
    steps = 0
    
    while not done:
        action = np.random.randint(0, 5)
        state, reward, done, info = env.step(action)
        episode_reward += reward
        steps += 1
    
    results['success'].append(info.get('success', False))
    results['steps'].append(steps)
    results['rewards'].append(episode_reward)
    
    if (episode + 1) % 10 == 0:
        print(f"Completed {episode + 1}/{num_test_episodes} episodes")

# Print statistics
print("\n=== Random Agent Performance ===")
print(f"Success rate: {np.mean(results['success'])*100:.1f}%")
print(f"Average steps: {np.mean(results['steps']):.1f} ± {np.std(results['steps']):.1f}")
print(f"Average reward: {np.mean(results['rewards']):.2f} ± {np.std(results['rewards']):.2f}")

if any(results['success']):
    success_steps = [s for s, succ in zip(results['steps'], results['success']) if succ]
    print(f"Average steps (successful): {np.mean(success_steps):.1f}")

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Success rate
axes[0].bar(['Success', 'Failure'],
            [np.mean(results['success']), 1 - np.mean(results['success'])],
            color=['green', 'red'], alpha=0.7)
axes[0].set_ylabel('Proportion')
axes[0].set_title('Success Rate (Random Agent)')
axes[0].set_ylim([0, 1])

# Steps distribution
axes[1].hist(results['steps'], bins=20, color='blue', alpha=0.7)
axes[1].axvline(np.mean(results['steps']), color='red', linestyle='--',
                label=f'Mean: {np.mean(results["steps"]):.1f}')
axes[1].set_xlabel('Steps')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Steps Distribution')
axes[1].legend()

# Reward distribution
axes[2].hist(results['rewards'], bins=20, color='purple', alpha=0.7)
axes[2].axvline(np.mean(results['rewards']), color='red', linestyle='--',
                label=f'Mean: {np.mean(results["rewards"]):.2f}')
axes[2].set_xlabel('Total Reward')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Reward Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

## 9. Summary

### ✅ What We've Demonstrated:

1. **Environment Setup**: Successfully created and initialized the visual search environment
2. **CIFAR-10 Integration**: Loaded dataset and placed objects on canvas
3. **Agent Observation**: Showed how agent sees limited 64x64 viewport
4. **Action Execution**: Demonstrated movement and "found" actions
5. **DINOv3 Features**: Extracted semantic features from observations
6. **Baseline Performance**: Measured random agent (~20% success rate)

### 🎯 Next Steps:

1. **Train DQN Agent**: Use reinforcement learning to learn optimal search strategy
2. **Leverage DINOv3**: Use visual features to guide search
3. **Evaluate Performance**: Compare trained agent vs random baseline
4. **Analyze Strategies**: Visualize learned search patterns

### 📊 Target Performance (After Training):

- Success Rate: **70%+** (vs 20% random)
- Average Steps: **<15** (vs ~35 random)
- Smart search patterns (move toward target)

---

**Continue to next notebook:** `02_visualize_dinov3.ipynb` (Coming soon)
